# പാഠം 11 - മോഡൽ കോൺടെക്സ്റ്റ് പ്രോട്ടോക്കോൾ (MCP)

The **മോഡൽ കോൺടെക്സ്റ്റ് പ്രോട്ടോക്കോൾ (MCP)** ഒരു ഓപ്പൺ സ്റ്റാൻഡേർഡാണ്, ഇത് ഏജന്റുകൾക്ക് റൺടൈമിൽ ടൂളുകൾ, വിഭവങ്ങൾ, ഡേറ്റാ സോഴ്‌സുകൾ ഡൈനാമിക്കായി കണ്ടെത്തി ഉപയോഗിക്കാൻ അനുയോജ്യമാണ്. ടൂളുകൾ എജന്റിലേക്ക് ഹാർഡ്‌കോഡ് ചെയ്യുന്നതിനുപകരം, MCP ഏജന്റുകൾക്ക് ആവശ്യാനുസരിച്ചു കഴിവുകൾ പ്രദർശിപ്പിക്കുന്ന ബാഹ്യ സർവറുകളുമായി കണക്ട് ചെയ്യാൻ ഇടയാക്കുന്നു.

In this lesson, you'll learn:
- MCP എന്താണ്, ഏജന്റ് സിസ്റ്റങ്ങളിലേക്ക് ഇത് എന്തുകൊണ്ട് പ്രാധാന്യമാണെന്ന്
- MCP ക്ലയന്റ്-സെർവർ ആർക്കിടെക്ചർ എങ്ങനെ പ്രവർത്തിക്കുന്നു
- MCP-സ്റ്റൈൽ ടൂൾ കണ്ടെത്തൽ ഉപയോഗിക്കുന്ന ഏജന്റുകൾ എങ്ങനെ നിർമ്മിക്കാമെന്ന്


## ക്രമീകരണം

**മുൻനിബന്ധനകൾ:**
- ഡിപ്ലോയുചെയ്‌ത മോഡലുള്ള Azure AI Foundry പ്രോജക്ട്
- പ്രാമാണീകരണത്തിന് `az login` ഓടിക്കുക


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) എന്താണ്?

MCP AI ഏജന്റുകൾക്ക് ബാഹ്യ ഉപകരണങ്ങളും ഡാറ്റാ സ്രോതസ്സുകളും കണ്ടെത്താനും അവയുമായി ഇടപെടാനും ഉപയോഗിക്കുന്ന ഒരു സ്റ്റാൻഡേർഡ് രീതിയാണ് നിർവചിക്കുന്നത്:

- **MCP Server**: സ്റ്റാൻഡേർഡ് പ്രോട്ടോക്കോളിലൂടെ ഉപകരണങ്ങളും വിഭവങ്ങളും പ്രോംപ്റ്റുകളും ലഭ്യമാക്കുന്നു
- **MCP Client**: സെർവറുകളുമായി ബന്ധപ്പെടുകയും ലഭ്യമായ കഴിവുകൾ കണ്ടെത്തുകയും ചെയ്യുന്ന ഏജന്റ് റൺടൈം
- **Dynamic Discovery**: ഏജന്റുകൾക്ക് ഹാർഡ്‌കോഡ് ചെയ്ത ഉപകരണങ്ങൾ ആവശ്യമില്ല — അവർ റണ്ണടൈമിൽ ലഭ്യമുള്ളവ കണ്ടെത്തും

ഈ സമീപനം ഏജന്റ് കോഡ് മാറ്റാതെ തന്നെ പുതിയ കഴിവുകൾ ചേർക്കാനുള്ള കഴിവുള്ള, വിപുലീകരിക്കാവുന്ന ഏജന്റ് സിസ്റ്റങ്ങൾ നിർമ്മിക്കാൻ വളരെ പ്രയോജനകരമാണ്.


## MCP എങ്ങനെ പ്രവർത്തിക്കുന്നു

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ഏജന്റ് (MCP ക്ലയന്റ്) ഒരു MCP സെർവറുമായി ബന്ധപ്പെടുന്നു
2. സെർവർ ലഭ്യമായ ഉപകരണങ്ങളുടെ പട്ടികയും അവയുടെ സ്കീമകളും മറുപടി നൽകുന്നു
3. ഏജന്റ് പിന്നീട് തന്റെ നിഗമനപ്രക്രിയയിൽ കണ്ടെത്തിയ ഏതൊരു ഉപകരണത്തെയും വിളിക്കാം
4. ഫലങ്ങൾ അതേ പ്രോട്ടോക്കോളിലൂടെ തിരികെ എത്തുന്നു


## MCP ടൂൾ കണ്ടെത്തലിന്റെ അനുകരണം

യഥാർത്ഥ MCP സെർവർ പ്രവർത്തനക്ഷമമായ ഒരു സെർവർ പ്രോസസ് ആവശ്യപ്പെടുന്നതിനാൽ, MCP-നോട് ബന്ധമുള്ളൊരു ആകോമഡേഷൻ സേവനം എന്ത് നൽകും എന്നു അനുകരിക്കുന്ന `@tool` ഫംഗ്ഷനുകൾ ഉപയോഗിച്ച് നമ്മൾ ഈ മാതൃക പ്രദർശിപ്പിക്കും.

പ്രൊഡക്ഷനിൽ, ഈ ടൂളുകൾ ലോക്കലായി നിർവചിക്കുന്നതിന് പകരം MCP സെർവറിൽ നിന്ന് ഡൈനാമിക്കായി കണ്ടുപിടിക്കപ്പെടും.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ശൈലിയിലെ ഉപകരണങ്ങളോടുകൂടിയ ഒരു ഏജന്റ് നിർമ്മിക്കൽ


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ഉത്പാദന പരിസ്ഥിതിയിൽ MCP

ഉത്പാദന പരിസ്ഥിതിയിൽ, MCP ശക്തമായ മാതൃകകൾ സാദ്ധ്യമാക്കുന്നു:

- **Dynamic tool discovery**: ഏജന്റുകൾ MCP സെർവറുകളുമായി ബന്ധപ്പെടുകയും റൺടൈമിൽ ടൂളുകൾ കണ്ടെത്തുകയും ചെയ്യുന്നു
- **Decoupled architecture**: ടൂൾ പ്രൊവൈഡറുകൾ ഏജന്റിൽ നിന്നു സ്വതന്ത്രമായി അപ്‌డേറ്റ് ചെയ്യാൻ കഴിയും
- **Cross-organization sharing**: ടീമുകൾ MCP സെർവറുകൾ വഴി വിശേഷതകൾ പുറത്തുവിടുകയും അവ ഏതെങ്കിലും ഏജന്റും ഉപയോഗിക്കാവുന്നതായി ചെയ്യുകയും ചെയ്യുന്നു
- **Microsoft Agent Framework support**: MAFയ്ക്ക് `mcp` ഇന്റഗ്രേഷൻ മുഖേന മുൻനിർമിച്ച MCP ക്ലയന്റ് പിന്തുണ ലഭ്യമാണ്

MAF-നൊപ്പം യഥാർത്ഥ MCP സെർവർ ഉപയോഗിക്കാൻ, നിങ്ങൾക്ക് `hosted_mcp_tool()` ഉപയോഗിച്ചോ MCP ക്ലയന്റ് ഇന്റഗ്രേഷൻ വഴിയോ ബന്ധപ്പെടാം.

**കൂടുതൽ അറിയുക:**
- [MCP സ്പെസിഫിക്കേഷൻ](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP പിന്തുണ](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:
- **MCP** ഏജന്റുകളും ഉപകരണ ദാതാക്കളും തമ്മിലുള്ള ഡൈനാമിക് ഉപകരണ കണ്ടെത്തലിനുള്ള ഒരു തുറന്ന സ്റ്റാൻഡേർഡാണ്
- **ക്ലയന്റ്-സർവർ ആർക്കിടെക്ചർ** ഏജന്റുകൾക്ക് റൺടൈമിൽ കഴിവുകൾ കണ്ടെത്താൻ അനുവദിക്കുന്നു
- MCP **വ്യാപകമാക്കാവുന്ന, വേർതിരിച്ച എജന്റ് സിസ്റ്റങ്ങൾ** സജ്ജമാക്കുന്നു, അവിടെ ഉപകരണങ്ങൾ കോഡ് മാറ്റങ്ങളില്ലാതെ ചേർക്കാൻ കഴിയും
- Microsoft Agent Framework ഉത്പാദന ഉപയോഗത്തിന് **ബിൽറ്റ്-ഇൻ MCP പിന്തുണ** നൽകുന്നു


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
അറിയിപ്പ്:
ഈ രേഖ AI വിവർത്തന സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ചാണ് വിവർത്തനം ചെയ്തത്. ഞങ്ങൾ കൃത്യതയ്ക്ക് ശ്രമിച്ചിരുന്നെങ്കിലും യാന്ത്രിക വിവർത്തനങ്ങളിൽ പിശകുകളും അപകൃതതകളും ഉണ്ടായിരിക്കാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. അതിന്റെ സ്വദേശഭാഷയിലുള്ള മൂല രേഖയാണ് അധികാരപരമായ ഉറവിടമായി കരുതേണ്ടത്. നിർണ്ണായകമായ വിവരങ്ങൾക്കായി പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം ശുപാർശ ചെയ്യുന്നു. ഈ വിവർത്തനം ഉപയോഗിച്ചതിലൂടെ ഉണ്ടാകുന്ന ഏതൊരു തെറ്റിദ്ധാരണയ്ക്കോ തെറ്റായ വ്യാഖ്യാനത്തിനോ ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
